# Putting the Pieces Together: Recreating the PHAT Andomeda Survey
---

## Introduction

In this webinar, you’ll:
- Use astroquery.mast to for search for observations in the MAST Archive
- Learn about MAST’s cloud archive that enables fast data access
- Create an HST mosaic of Andromeda


### Background

The [Panchromatic Hubble Andromeda Treasury (PHAT)](https://archive.stsci.edu/hlsp/phat) was a multi-cycle HST program, designed to efficiently map a third of of the star-forming disk of Andromeda.

![phat brick coverage](phat_coverage.png)

The "efficiency" here comes from using a unique HST observing mode: [parallel observations](https://stsci.recollectcms.com/nodes/view/5271). In short, all HST instruments lie in the same focal plane, so it is possible to use more than one instrument at a time. PHAT takes advantage of this multi-instrument capability to observe in different wavelengths. Even with this clever design, the spatial extent of Andromeda on the sky is quite large. For planning purposes, this program was broken into 23 sections the team named "bricks" (shown above in blue rectangles). Each brick consists of 18 individual HST pointings, taken with the WFC3 and ACS instruments. Today, we'll be creating a mosaic of one of these bricks!

## Imports

We'll need a few packages to accomplish our goal:
* `astropy`, for reading in files, handling astronomy units, and setting a reasonable contrast for our mosaic
* `astroquery` to run our MAST query
* `reproject` to assemble our mosaic

In [ ]:
# run our query
from astroquery.mast import Observations

# read files, use astronomy units,
from astropy.io import fits
from astropy.visualization import ZScaleInterval
import astropy.units as u

from reproject import reproject_interp
from reproject.mosaicking import find_optimal_celestial_wcs, reproject_and_coadd

# display our nice image
import matplotlib.pyplot as plt

import numpy as np

## Running a MAST Query: the Basics

There are multiple ways to search MAST. We offer both search forms (e.g. the [HST form](https://mast.stsci.edu/search/ui/#/hst)) and application programming interfaces (APIs). In this tutorial, we'll be using the [astroquery.mast](https://astroquery.readthedocs.io/en/latest/mast/mast.html) Python API to search. 

Before we begin constructing a query for our PHAT Andromeda data, let's cover the basics of searching for MAST data. Every query begins with a set of conditions, like "JWST observations of Trappist-1". To turn this into an API query, we need to understand the queryable parameters. You can find these parameters on the [CAOM field descriptions](https://mast.stsci.edu/api/v0/_c_a_o_mfields.html) page. In a notebook, you can also run `Observations.get_metadata('observations')`

In [ ]:
# show query options


This is quite useful, if a bit overwhelming. For this notebook, we'll focus on a few key parameters:

| Criterion | Meaning |
|--|--|
|`object_name`| The name of the celestial object: this name will be resolved to coordinates|
|`target_name`| The name of the celestial object **as entered by the proposer**|
|`obs_collection`| Roughly equivalent to mission |
|`proposal_id` | For HST and JWST, the assigned proposal number |
|`wave_region` | The wavelength region of the observations (e.g., `UV` or `optical`)

Using the `object_name` criterion can sometimes be slow since the API must:
1. Query a resolver service ([likely Simbad or NED](https://mastresolver.stsci.edu/Santa-war/)) to transform your input into coordinates
2. Look for observations with a spatial match on your input. This is no small task when the database has over 500 million rows!

By default, our spatial searches have a radius of 0.2 degrees. Searches with large radii are more likely to be slow; unless you have a known reason to adjust the radius, this default will likely work for your search.

To run our search, we'll use the `Observations.query_criteria` function with the above criteria. Let's turn "JWST observations of Trappist-1" into a query:

In [ ]:
# run a query for JWST Trappist-1 observations
obs = 

len(obs)

Despite calling this search "slow", it returns results relatively quickly.

### A Brief Aside: Object vs. Target

We mentioned that searching on `target_name` is typically faster. So why not use `target_name` to make every search faster?

The easiest way to answer this is to look at the `target_name` list from our search above. Remember, this value is generated by PIs. Let's take a look at what they called the targets:

In [ ]:
# get the unique set of target names


Whoops.

These results highlight the pitfalls of searching by target name. Sometimes an observer may use a non-standard name, or a the name might not be populated in the database at all!

We can sidestep these issues (on occasion) by using a wildcard character.

One useful trick for searching by `target_name` is the use of a wildcard character. In MAST, you can use an asterisk as a [glob wildcard](https://en.wikipedia.org/wiki/Glob_(programming)). This is the standard for all Unix-like operating systems (including MacOS).

| Example String | Matches | Does not match |
|--|--|--|
|`law*`| `law`, `laws`, `law school is quite expensive` | `la`, `aw`, `los angeles` |
|`*law*` | `law`, `unlawful`, `what are you, the law police?` | `la`, `w`, `trappist-1`|

TO-DO: repeat the above but with `target_name` and a wildcard character. {show an example of a wildcard match before, just in text}

Exercise: repeat the search for JWST observations of Trappist-1, but this time with `target_name`. Using a wildcard, you should be able to recover 388 observations.

In [ ]:
# wildcard search for trappist-1
obs = 

len(obs)

Ultimately, this is quite fast, but but we're still missing 17 observations. This is why it's typically safest to use `object_name` for a resolved search.

## Searching for a Single PHAT "Brick"

With those caveats out of the way, we can carefully use the `target_name` trick for fast searches of our PHAT data.

In [ ]:
# our full brick 01 query
obs = 

obs

This looks good to m—

> "Hey wait a minute. You said each brick has 18 observations! I see 19."

Good catch. One of our sub-fields is duplicated. This won't have a major effect on our mosaic, but as an exercise for the reader: Can you figure out which one?

ans: (uǝʌǝs ʎʇuǝʍʇ ɟo ʇooɹ ǝqnɔ) ɹǝqɯnu

## Cloud Access of MAST Products

We're all running this notebook on TIKE, which is conveniently located in the same datacenter as all of MAST's cloud data. This means that your internet speed is irrelevant; only the *commands* will be sent to the server. Everything else will be processed as though it were local. It really is like you plugged your computer into MAST!

Our work so far has given us a set of Observations. What we would really to access is the underlying files. In MAST, we call these "products". They can be accessed using the `get_product_list` function:

In [ ]:
# get our list of products

prod

That's a lot of files! We can filter them down further with a call to `filter_products`. We'll use [drizzled](https://en.wikipedia.org/wiki/Drizzle_(image_processing)) images, since the drizzling algorithm produces high-quality images that are nice to look at. We'll also filter for "minimum recommended products": essentially the products that we have identified as being most useful for science. Trust us at your own risk!

In [ ]:
# filter our products down to drizzled / MRP products

filt

This gets us down to our 18 observations (+1 duplicated field). Our last step is to figure out where these files live in the cloud.

In [ ]:
# get cloud URIs for our files

curis

These are URIs. You type URLs into a browswer, but URIs are more generic. If you've ever seen `mailto:email@email.com`, you've seen a URI using the `mailto` scheme. Our files are using the `s3` scheme; this is how they're organized in the cloud.

We could have done this in a single step with the `Observations.get_cloud_uris` function, but it's important to understand each step of the process.

### Open a File

Now that we have the locations of our files, we can open them with a standard `fits.open()` call. You can run this on your local machine, although you will run into bandwidth limits if you do.

We'll open all of the files at once, for later convenience. Plus we get to use a fun Python syntax!

In [ ]:
# need "anon":true to anonymously access the cloud files


Let's test that this worked by making a plot.

In [ ]:
# Create a figure on which to plot our data
fig = plt.figure(0, [9, 9])
ax = fig.add_subplot(111)

# Get the primary data from the first fits file


# Automatically scale the brightness based on the data
interval = ZScaleInterval(contrast=0.4)
lims = interval.get_limits(test_data)

# Show our data with the scaling from above
ax.imshow(test_data, vmin=lims[0], vmax=lims[1], cmap='grey')

The faint line through the middle of the image is a result of the drizzling process.

## Mosaic Building

Unfortunately, the `Reproject` package does not understand the `s3` scheme and thus cannot read the files from their cloud locations. That's ok, since we can talk about how to read files into our memory, which will apply to many workflows! Since we're on TIKE, this will still be much faster than trying to download the files to your local machine.

Reproject is expecting a particular format of `(data, header)` for each of our observations. Yay for more Python list comprehension!

In [ ]:
# format our data so reproject will be happy
hdu_tuples = [(hdu[1].data, hdu[1].header) for hdu in hdus]

Our data preparation is complete.

To combine our images, we need to start by understanding what our output will look like. We'll use `find_optimal_celestial_wcs` to select a reasonable "world coordinate system" for image. Without WCS, we could make an image; however, it would not be aligned to celestial north. 

We also need to decide how high-resolution our image should be. Although it is great fun to create 8k images, we'll try to be reasonable in this example. Let's set our resolution to one quarter of an arcsecond.

In [ ]:
# generate wcs and shape of our final mosaic
wcs, shape = find_optimal_celestial_wcs(hdu_tuples, 
                                        resolution=0.25*u.arcsec
                                       )

We'll use `wcs` later to align our image.

The `shape` tells us the size of our image. Did we choose a reasonable resolution?

In [ ]:
shape

Our image is almost 4k! We're not fully utilizing HST's exquisite sensitivity, but we should still get a nice moasic. 

The `reproject_and_coadd` function will assemble our mosaic for us.
Let's run the code to assemble it:

In [ ]:
# get the array (brightness values) and footprint of our image
array, footprint = reproject_and_coadd(hdu_tuples,
                                       wcs, shape,
                                       reproject_function=reproject_interp,
                                       match_background=True
                                      )

One last check before we look at our nice image: do we have a reasonable overlapping footprint?

In [ ]:
# plot the footprint with a labeled colorbar
plt.imshow(footprint)
plt.colorbar()

What we're looking at is, in effect, the number of times HST observed each region. Our mosaic includes a `0` for locations that have no data from any of our observations. We don't particularly want this, so we can create a mask using our footprint.

In [ ]:
masked_data = np.ma.masked_array(array, footprint==0)

Now that we have a maks to hide the regions with no data, we can create our figure.

In [ ]:
# Create the figure for the new map
fig = plt.figure(0, [9, 11])
ax = fig.add_subplot(111, projection=wcs)

# Automatically scale the brightness
interval = ZScaleInterval(contrast=0.4)
lims = interval.get_limits(masked_data)

# Plot the new map
ax.imshow(masked_data, vmin=lims[0], vmax=lims[1], origin='lower', cmap='grey')
ax.set_xlabel('RA')
ax.set_ylabel('Dec')

Very nice! We can see some of the dust structure near the center of andromeda.

If you like this mosaic, you can un-comment the cell below to save/download the image. The [full mosaic](https://science.nasa.gov/asset/hubble/hubble-m31-phatphast-mosaic/) is availble on NASA's website.

In [ ]:
# plt.imsave(fname='brick1-mosaic.png', arr=masked_data, vmin=lims[0], vmax=lims[1], origin='lower', cmap='grey')

## About this Notebook

This notebook was written for the 2026 MAST Summer webinar series.

**Author:** Thomas Dutkiewicz<br>
**Keywords:** HST, mosaic, PHAT <br>

***
[Top of Page](#top)
<img style="float: right;" src="https://raw.githubusercontent.com/spacetelescope/style-guides/master/guides/images/stsci-logo.png" alt="Space Telescope Logo" width="200px"/> 